# 22장 · 정답 쪽을 정한 뒤 검색 순위 읽기

“검색이 좋아졌다”를 무엇으로 확인할까요? 모델이 친절한 답을 써도 필요한 설명 쪽을 검색하지 못했다면 다음 질문에서 다시 실패할 수 있습니다. 먼저 사람이 원문에서 정답 페이지를 정한 뒤, 답변 생성을 끄고 검색 결과만 대조합니다.

## 실습 준비

전체 실습 묶음을 풀고 노트북을 엽니다. 같은 폴더에서 시작해 자료 묶음의 `pre-mid/documents`를 읽습니다. 공용 준비 파일에 뒤에서 사용할 함수도 있으므로 **아래 셀에서 가져오는 이름**부터 살펴보세요. 처음 한 번 커널에서 `%pip install pypdf langchain-openai langchain-core chromadb python-dotenv`를 실행하고 커널을 재시작하세요. 작업 폴더의 .env에 OPENAI_API_KEY를 둡니다. 첫 색인 생성과 새 질문 임베딩은 API를 호출합니다.

In [1]:
from pathlib import Path
import sys

# 압축을 푼 폴더 또는 그 아래에서 실습 자료의 위치를 찾습니다.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("전체 실습 묶음을 풀고 pre-mid 폴더 옆에서 실행하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))

## 먼저 손으로 두 문제 풀기

정답 쪽 하나가 있는 질문 두 개를 가정합니다. 첫 질문에서는 정답이 2위, 두 번째에서는 3위입니다. 두 후보까지만 다음 단계로 보낸다면 첫 질문만 성공합니다. **Recall@K**는 정답으로 정한 쪽 중 앞 K개 안에 들어간 비율입니다. 여기서 각 질문의 정답은 하나이므로 1 또는 0입니다. 여러 쪽이 모두 필요하면 찾은 정답 쪽 개수를 전체 정답 쪽 개수로 나눕니다.

**역순위**는 첫 정답의 순위가 r일 때 1/r입니다. 앞 K개 안에 없으면 0입니다. 여러 질문의 역순위를 평균 내면 **MRR@K**입니다. 위 두 질문에서 K=2일 때 평균 Recall은 (1+0)/2=0.5, MRR은 (1/2+0)/2=0.25입니다. K를 3으로 바꾸면 각각 어떻게 될지 먼저 계산하세요.

In [2]:
def page_scores(ranked_ids, relevant_ids, k):
    # 정답 목록은 검색을 끝낸 뒤 채점에만 씁니다.
    top = ranked_ids[:k]
    hits = set(top) & relevant_ids
    recall = len(hits) / len(relevant_ids)
    reciprocal = next((1 / pos for pos, key in enumerate(top, 1)
                       if key in relevant_ids), 0)
    return recall, reciprocal

a = page_scores(["x", "a", "y"], {"a"}, 3)
b = page_scores(["x", "y", "b"], {"b"}, 3)
print("Recall@3", (a[0]+b[0])/2, "MRR@3", round((a[1]+b[1])/2, 3))

Recall@3 1.0 MRR@3 0.417


## 실제 질문으로 기준 결과 남기기

자료의 questions/recorder-dev.csv에는 질문만 있습니다. 팀원이 정답 쪽을 손으로 찾고 서로 합의한 뒤 answers/recorder-dev.csv와 대조합니다. 아직 answers/recorder-holdout.csv는 열지 마세요. 마지막 검증에 쓸 새 문제입니다.

In [3]:
import csv
from rag_lab import DOCUMENTS, open_index, retrieve
def read_csv(path):
    with path.open(encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))
questions = read_csv(DOCUMENTS / "questions/recorder-dev.csv")
gold = {row["id"]: row for row in read_csv(DOCUMENTS / "answers/recorder-dev.csv")}
index = open_index()
k = 3
scores = []
for item in questions:
    ranked = [r["id"] for r in retrieve(index, item["question"], k)]
    expected = gold[item["id"]]
    if expected["expected_action"] == "answer":
        key = f"{expected['source']}#p{expected['page']}"
        recall, rr = page_scores(ranked, {key}, k)
        scores.append((recall, rr))
        print(item["id"], ranked, "정답", key, "Recall", recall, "역순위", rr)
    else:
        print(item["id"], "근거 없음: 답변 보류를 별도로 확인")

D01 ['recorder-a-1.1.pdf#p1', 'recorder-a-1.0.pdf#p1', 'recorder-b-1.0.pdf#p1'] 정답 recorder-a-1.1.pdf#p1 Recall 1.0 역순위 1.0
D02 ['recorder-b-1.0.pdf#p1', 'recorder-a-1.1.pdf#p1', 'recorder-a-1.0.pdf#p1'] 정답 recorder-b-1.0.pdf#p1 Recall 1.0 역순위 1.0
D03 ['recorder-a-1.0.pdf#p2', 'recorder-a-1.1.pdf#p2', 'recorder-b-1.0.pdf#p2'] 정답 recorder-a-1.0.pdf#p2 Recall 1.0 역순위 1.0
D04 ['recorder-a-1.1.pdf#p2', 'recorder-a-1.0.pdf#p2', 'recorder-a-1.0.pdf#p1'] 정답 recorder-a-1.1.pdf#p2 Recall 1.0 역순위 1.0
D05 근거 없음: 답변 보류를 별도로 확인


In [4]:
print("질문 수", len(scores))
print("평균 Recall", round(sum(s[0] for s in scores)/len(scores), 3))
print("MRR", round(sum(s[1] for s in scores)/len(scores), 3))

질문 수 4
평균 Recall 1.0
MRR 1.0


D05에는 정답 페이지가 없습니다. 빈 정답 목록을 억지로 분모에 넣지 않고 21장의 보류 기준으로 별도 판정합니다. 수치 옆에는 적어도 한 질문의 실제 순위를 남겨 두세요. 똑같이 MRR이 낮아도 정답이 2위에 있는 경우와 아예 검색되지 않은 경우의 해결책은 다릅니다.

## 하나라도 맞았는지와 모두 찾았는지는 다르다

정답 하나가 있으면 적중(Hit)과 Recall 값이 같아 보입니다. 두 판본 비교에서는 ‘하나라도 검색됨’이 1이어도 근거의 반쪽만 찾을 수 있습니다. 아래의 두 학생이 같은 점수를 받는 기준과 서로 다른 점수를 받는 기준을 나눠 보세요.

In [5]:
relevant = {"3월판 2쪽", "4월판 2쪽"}
for rows in [["3월판 2쪽", "다른 쪽"], ["3월판 2쪽", "4월판 2쪽"]]:
    hit = int(bool(set(rows) & relevant))
    recall, rr = page_scores(rows, relevant, 2)
    print(rows, "Hit", hit, "Recall", recall, "첫 정답 역순위", rr)

['3월판 2쪽', '다른 쪽'] Hit 1 Recall 0.5 첫 정답 역순위 1.0
['3월판 2쪽', '4월판 2쪽'] Hit 1 Recall 1.0 첫 정답 역순위 1.0


둘 다 첫 번째에 정답이 있으므로 Hit와 역순위는 1입니다. 두 날짜를 비교할 수 있다는 목표에서는 모든 필요한 쪽을 모은 두 번째 검색만 충분합니다. 질문 하나에 붙인 관련 문서 목록을 불완전하게 만들면 평가 지표를 정확히 계산해도 팀의 판단이 틀릴 수 있습니다.

동료가 쓴 질문의 표현을 정답에서만 볼 수 있는 단어로 바꿔 검색하면 평가가 쉬워집니다. 실제 사용자의 원문 질문은 그대로 보관하고, 사람이 붙인 정답 페이지는 계산 단계에서만 펼치세요. 개선을 위해 여러 번 본 개발 질문과 마지막 확인용 질문도 구분합니다.

## 실습 · 첫 정답과 필요한 모든 쪽 구분하기

“A형 3월판과 4월판의 파일 분할 시간을 비교해 주세요”에는 각 판본의 2쪽이 모두 필요합니다. 두 후보를 정해 두 쪽 중 한 쪽만 1위에 있을 때 점수를 먼저 손으로 구하고 함수에 넣어 확인하세요. 다음에는 실제 질문으로 검색해 어느 판본이 빠졌는지 확인합니다.

In [6]:
# page_scores에 두 정답 ID를 집합으로 넘겨 보세요.

<details class="notebook-answer"><summary>모범 풀이와 판정 기준</summary><pre><code>keys = {&quot;recorder-a-1.0.pdf#p2&quot;,&quot;recorder-a-1.1.pdf#p2&quot;}
example = [&quot;recorder-a-1.1.pdf#p2&quot;,&quot;recorder-b-1.0.pdf#p2&quot;]
print(page_scores(example, keys, 2))
query = &quot;A형의 3월판과 4월판 파일 분할 간격은 어떻게 달라졌나요?&quot;
ranked = [r[&quot;id&quot;] for r in retrieve(index, query, k=4)]
print(ranked, page_scores(ranked, keys, 4))</code></pre><p>손계산의 Recall@2는 0.5, 첫 정답 역순위는 1입니다. 첫 번째 정답이 맨 앞에 있어도 두 판본을 모두 비교할 근거는 부족합니다.</p><p>집필 시 실행 결과</p><pre>(0.5, 1.0)
[&#x27;recorder-a-1.0.pdf#p2&#x27;, &#x27;recorder-a-1.1.pdf#p2&#x27;, &#x27;recorder-b-1.0.pdf#p2&#x27;, &#x27;recorder-a-1.0.pdf#p1&#x27;] (1.0, 1.0)</pre></details>

## 독립 실습 · 정답 페이지 두 개가 모두 필요한 비교

“2027년 3월과 4월 사이에 A형의 파일 분할 간격이 얼마나 늘었나요?”라는 별도 질문을 추가합니다. 두 날짜의 정답을 직접 찾아 페이지 ID를 붙입니다. 정답 하나만 1위에 들어간 검색 결과와 두 쪽 모두 들어간 검색 결과를 만드세요. K=2에서 각각 Recall과 역순위를 계산하고, 15분이라는 차이를 답할 수 있는 쪽을 고르세요.

<details class="notebook-answer"><summary>판정 예시와 해설</summary><p>상위 두 쪽 중 1위만 정답이면 Recall@2는 1/2, 역순위는 1입니다. 두 쪽이 모두 정답이면 Recall@2와 역순위가 각각 1입니다. 후자의 두 쪽을 열면 45분과 30분의 차이가 15분이라는 비교를 뒷받침할 수 있습니다. 전자에 15분이라고 답하는 일은 검색된 자료 하나로 설명할 수 없습니다.</p></details>

## 다음 수업에 가져갈 것

개발 질문의 기준 후보·정답·순위를 저장합니다. 정답이 후보 안에 있는 질문과 후보 밖에 있는 질문을 나눠 다음 장에서 재정렬로 무엇을 개선할 수 있는지 확인하세요.